In [0]:

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Create Spark session
spark = SparkSession.builder \
    .appName("QuarterlyAverageSales") \
    .getOrCreate()

# 2. Create dummy sales data
data = [
    (101, "2026-01-10", 1000),
    (101, "2026-02-15", 2000),
    (101, "2026-03-20", 3000),
    (101, "2026-04-10", 4000),
    (101, "2026-05-15", 5000),
    (101, "2026-06-20", 6000),
    (101, "2026-07-10", 7000),
    (101, "2026-08-15", 8000),
    (101, "2026-09-20", 9000),
    (102, "2026-01-05", 500),
    (102, "2026-02-10", 1500),
    (102, "2026-03-15", 2500),
    (102, "2026-04-05", 3500),
    (102, "2026-05-10", 4500),
    (102, "2026-06-15", 5500)
]

columns = ["sale_id", "sales_date", "sales_amount"]

df = spark.createDataFrame(data, columns)



In [0]:
quarterly_avg_sales_df = df.withColumn("sales_date",F.to_date('sales_date')) \
                           .withColumn("year",F.year(F.col("sales_date"))) \
                           .withColumn("quarter",F.quarter(F.col("sales_date"))) \
                           .groupBy('sale_id','year','quarter') \
                            .agg(F.avg("sales_amount").alias("avg_sales_amount"))
quarterly_avg_sales_df.show()

In [0]:
windowSpec = Window.partitionBy('sale_id').orderBy('year','quarter')

final_df = quarterly_avg_sales_df.withColumn("prev_avg_sales",F.lag("avg_sales_amount",1).over(windowSpec)) \
                                 .withColumn("sales_diff",F.round(F.col("avg_sales_amount")-F.col("prev_avg_sales"),2))
final_df.show()